# Home Credit Default Risk — GCI 2026 Competition

**Approach**
- **Data:** only the provided `train.csv` / `test.csv` (no external data, no hand-labeling).
- **Cleaning:** `DAYS_EMPLOYED = 365243` replaced by NaN with an anomaly flag; `OWN_CAR_AGE ≥ 60` and `ORGANIZATION_TYPE = XNA` treated as missing.
- **Features (87):** lender-style ratios (annuity/income, credit/income, loan term, income left after payments), time ratios relative to age, `EXT_SOURCE` aggregates and interactions, and value-frequency counts.
- **Categoricals:** kept as categories and handled natively by LightGBM and CatBoost (no one-hot).
- **Models:** regularized LightGBM (3 seeds × 5-fold Stratified CV), CatBoost (5-fold Stratified CV) and XGBoost (5-fold Stratified CV).
- **Ensemble:** rank average of the three models; the weights are chosen on out-of-fold AUC (small grid search), never on the leaderboard.
- **Reproducibility:** all random seeds fixed; *Restart & Run all* reproduces `output/submission.csv`.

**Experiment log (5-fold OOF AUC)**

| Step | CV AUC | Public |
|---|---|---|
| Tutorial baseline (5 features, RF) | 0.669 | 0.668 |
| All numeric features + median fill | 0.730 | 0.731 |
| One-hot categoricals, LR + RF average | 0.738 | 0.742 |
| LightGBM, 5-fold CV, first engineered features | 0.760 | 0.764 |
| 3-seed LightGBM + CatBoost (one-hot data) | 0.7624 | 0.7644 |
| 87 features, native categoricals, regularized LightGBM + CatBoost (previous version) | 0.7656 | 0.7682 |
| **This notebook:** same features, plus XGBoost as a third model, blend weights chosen on OOF | 0.7657 | 0.7681 |

Tested and dropped (no gain in CV): group-relative statistics, out-of-fold target encoding, extra count features, EXT polynomial and rank features, extra ratio features, a shallow LightGBM and a logistic regression.

**Credits:** several feature ideas (income left, overpay ratio, EXT interactions, value-frequency counts) were adapted from the public repository
[diwanparker/gci-world-competition](https://github.com/diwanparker/gci-world-competition) and common public Home Credit Kaggle notebooks.
The experiments and the CV / blending code were done with the help of Claude (AI assistant) and reviewed by me. The XGBoost model and the OOF-based blend weights were added on 19 Nov with the same help, and I reviewed that code too.

## 0. Setup
This cell installs CatBoost, loads the libraries and finds the data folder.
On Colab it mounts Google Drive first. On a normal computer it uses the current folder.

In [1]:
!pip install -q catboost

In [2]:
import os, time, zipfile, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostClassifier
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

# Paths: Google Drive on Colab, current folder elsewhere
if os.path.exists("/content"):
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/GCI/Competition")
current_dir = Path(os.getcwd())
INPUT_DIR = current_dir / "input"
# Stop early if the data files are not where we expect them
assert (INPUT_DIR / "train.csv").exists() and (INPUT_DIR / "test.csv").exists()

## 1. Feature engineering
Only the given `train.csv` and `test.csv` are used. The steps are:
1. `base_features`: clean a few odd values and build ratio features. Each row uses only its own values.
2. `extra_features`: more row-wise ratios and `EXT_SOURCE` interactions.
3. `count_features`: how often a value appears in train and test together. It uses feature values only, never the target.
4. `to_category` and `clean_inf`: set category types and replace infinite values with NaN.

In [3]:
import numpy as np
import pandas as pd

# Categorical columns (kept as categories, no one-hot)
CAT_COLS = ["NAME_CONTRACT_TYPE", "CODE_GENDER", "NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE",
            "NAME_FAMILY_STATUS", "NAME_HOUSING_TYPE", "OCCUPATION_TYPE", "ORGANIZATION_TYPE"]
# The three external score columns
EXT = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
# Credit bureau request counts
REQ = ["AMT_REQ_CREDIT_BUREAU_HOUR", "AMT_REQ_CREDIT_BUREAU_MON",
       "AMT_REQ_CREDIT_BUREAU_QRT", "AMT_REQ_CREDIT_BUREAU_YEAR"]


def load(d="input"):
    """Read the two csv files. Return train features, test features and the target."""
    tr = pd.read_csv(f"{d}/train.csv")
    te = pd.read_csv(f"{d}/test.csv")
    y = tr["TARGET"].values
    return tr.drop(columns=["TARGET", "SK_ID_CURR"]), te.drop(columns=["SK_ID_CURR"]), y


def base_features(df):
    """Row-wise features (each row uses only its own values)."""
    df = df.copy()
    f = {}
    # 365243 is a placeholder for "not employed": flag it, then treat it as missing
    f["DAYS_EMPLOYED_ANOM"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)
    df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)
    # Car age of 60 or more is not realistic, so treat it as missing
    df.loc[df["OWN_CAR_AGE"] >= 60, "OWN_CAR_AGE"] = np.nan
    f["NAN_COUNT"] = df.isna().sum(axis=1)  # how many values are missing in this row

    inc, cred, ann, goods = df["AMT_INCOME_TOTAL"], df["AMT_CREDIT"], df["AMT_ANNUITY"], df["AMT_GOODS_PRICE"]
    fam = df["CNT_FAM_MEMBERS"].clip(lower=1)
    # Money ratios (like a lender would look at)
    f["ANNUITY_INCOME"] = ann / inc
    f["CREDIT_INCOME"] = cred / inc
    f["CREDIT_TERM"] = ann / cred
    f["GOODS_CREDIT"] = goods / cred
    f["CREDIT_GOODS_DIFF"] = cred - goods
    f["INCOME_PER_PERSON"] = inc / fam
    f["CHILDREN_RATIO"] = df["CNT_CHILDREN"] / fam
    f["ANNUITY_PER_PERSON"] = ann / fam

    # Time features, relative to the age of the person
    age = -df["DAYS_BIRTH"] / 365.25
    f["AGE"] = age
    f["EMPLOYED_AGE"] = df["DAYS_EMPLOYED"] / df["DAYS_BIRTH"]
    f["ID_AGE"] = df["DAYS_ID_PUBLISH"] / df["DAYS_BIRTH"]
    f["REG_AGE"] = df["DAYS_REGISTRATION"] / df["DAYS_BIRTH"]
    f["PHONE_AGE"] = df["DAYS_LAST_PHONE_CHANGE"] / df["DAYS_BIRTH"]
    f["PHONE_EMPLOYED"] = df["DAYS_LAST_PHONE_CHANGE"] / df["DAYS_EMPLOYED"]
    f["CREDIT_PER_AGE"] = cred / age

    # Summary of the three EXT_SOURCE scores
    e = df[EXT]
    em = e.mean(axis=1)
    f["EXT_MEAN"] = em
    f["EXT_MIN"] = e.min(axis=1)
    f["EXT_MAX"] = e.max(axis=1)
    f["EXT_STD"] = e.std(axis=1)
    f["EXT_PROD"] = df["EXT_SOURCE_1"] * df["EXT_SOURCE_2"] * df["EXT_SOURCE_3"]
    f["EXT_NANS"] = e.isna().sum(axis=1)
    f["EXT2x3"] = df["EXT_SOURCE_2"] * df["EXT_SOURCE_3"]
    f["EXT_MEAN_TERM"] = em * f["CREDIT_TERM"]
    f["REQ_SUM"] = df[REQ].sum(axis=1, min_count=1)
    # "XNA" means unknown organization, so treat it as missing
    df["ORGANIZATION_TYPE"] = df["ORGANIZATION_TYPE"].replace("XNA", np.nan)
    return pd.concat([df, pd.DataFrame(f, index=df.index)], axis=1)


def extra_features(df):
    """More row-wise features (ideas from diwanparker/gci-world-competition)."""
    f = {}
    inc, cred, ann, goods = df["AMT_INCOME_TOTAL"], df["AMT_CREDIT"], df["AMT_ANNUITY"], df["AMT_GOODS_PRICE"]
    # Loan length in months (credit / annuity), also rounded to a whole number
    months = cred / ann
    term_round = months.round()
    f["CREDIT_MONTHS_ROUND"] = term_round
    f["INCOME_LEFT"] = inc - 12.0 * ann              # income left after one year of payments
    f["GOODS_INCOME"] = goods / inc
    f["INCOME_PER_CHILD"] = inc / (df["CNT_CHILDREN"] + 1)
    f["OVERPAY_RATIO"] = ann * term_round / cred     # total paid back compared with the credit
    f["TOTAL_PAY_OVER_GOODS"] = ann * term_round / goods
    f["CAR_AGE_RATIO"] = df["OWN_CAR_AGE"] / df["AGE"]
    f["AGE_AT_ID"] = (df["DAYS_ID_PUBLISH"] - df["DAYS_BIRTH"]) / 365.25
    f["CREDIT_END_AGE"] = df["AGE"] + months / 12.0  # age when the loan ends
    # More EXT_SOURCE features: median, weighted mean, products, differences
    e1, e2, e3 = df["EXT_SOURCE_1"], df["EXT_SOURCE_2"], df["EXT_SOURCE_3"]
    em = df["EXT_MEAN"]
    f["EXT_MEDIAN"] = df[EXT].median(axis=1)
    f["EXT_WEIGHTED"] = (2 * e1.fillna(em) + e2.fillna(em) + 3 * e3.fillna(em)) / 6
    f["EXT1x2"], f["EXT1x3"] = e1 * e2, e1 * e3
    f["EXT1m2"], f["EXT1m3"], f["EXT2m3"] = e1 - e2, e1 - e3, e2 - e3
    for k, col in [(1, e1), (2, e2), (3, e3)]:
        f[f"EXT{k}_AGE"] = col / df["AGE"]
        f[f"EXT{k}_MONTHS"] = col * months
    f["EXT_MEAN_CREDIT_INC"] = em * df["CREDIT_INCOME"]
    f["EXT_MEAN_ANN_INC"] = em * df["ANNUITY_INCOME"]
    return pd.concat([df, pd.DataFrame(f, index=df.index)], axis=1)


# Columns (or pairs of columns) for which we count how often the value appears
COUNT_KEYS = [("AMT_CREDIT", "AMT_ANNUITY"), ("AMT_GOODS_PRICE",), ("CREDIT_MONTHS_ROUND",),
              ("AMT_INCOME_TOTAL",), ("DAYS_BIRTH",)]


def count_features(tr, te):
    """How common a value is (frequency). Uses feature values of train+test, never labels."""
    both = pd.concat([tr, te], ignore_index=True)  # count over train and test together
    ft, fe = {}, {}
    for keys in COUNT_KEYS:
        name = "CNT_" + "_".join(keys)
        cnt = both.groupby(list(keys), dropna=False)[keys[0]].transform("size").values
        ft[name], fe[name] = cnt[: len(tr)], cnt[len(tr):]  # split back into train and test
    return (pd.concat([tr, pd.DataFrame(ft, index=tr.index)], axis=1),
            pd.concat([te, pd.DataFrame(fe, index=te.index)], axis=1))


def to_category(tr, te):
    """Make text columns 'category' type. Test uses the same categories as train."""
    tr, te = tr.copy(), te.copy()
    for c in CAT_COLS:
        tr[c] = tr[c].astype("category")
        te[c] = pd.Categorical(te[c], categories=tr[c].cat.categories)
    return tr, te


def clean_inf(tr, te):
    """Replace infinite values (from divisions by zero) with NaN."""
    return tr.replace([np.inf, -np.inf], np.nan), te.replace([np.inf, -np.inf], np.nan)

### Build the final feature tables
Run the steps in order. The result has 87 features for both train and test.

In [4]:
tr0, te0, y = load(INPUT_DIR)
sample_sub = pd.read_csv(INPUT_DIR / "sample_submission.csv")

A_tr, A_te = base_features(tr0), base_features(te0)      # step 1
B_tr, B_te = extra_features(A_tr), extra_features(A_te)  # step 2
X, X_test = clean_inf(*to_category(*count_features(B_tr, B_te)))  # steps 3 and 4

# Train and test must have exactly the same columns in the same order
assert list(X.columns) == list(X_test.columns)
print("train", X.shape, "| test", X_test.shape, "| default rate", round(y.mean(), 4))

train (171202, 87) | test (61500, 87) | default rate 0.0807


## 2. LightGBM — 3 seeds × 5-fold Stratified CV
Strong regularization (`min_child_samples=300`, `colsample_bytree=0.3`, `reg_lambda=20`) gave the best CV in the parameter sweep.

How it works: for each seed, the training data is split into 5 folds. A model is trained on 4 folds and
predicts the 5th fold (these are the out-of-fold, or OOF, predictions). The same model also predicts the test set.
Early stopping picks the number of trees using the validation fold.

In [5]:
LGB_PARAMS = dict(n_estimators=20000, learning_rate=0.02, num_leaves=31, min_child_samples=300,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.3, reg_alpha=1.0,
                  reg_lambda=20.0, verbose=-1)
LGB_SEEDS = [0, 1, 2]

oof_lgb = np.zeros(len(X))        # out-of-fold predictions for train
pred_lgb = np.zeros(len(X_test))  # averaged predictions for test
for seed in LGB_SEEDS:
    t = time.time()
    oof_seed = np.zeros(len(X))
    for tr_idx, va_idx in StratifiedKFold(5, shuffle=True, random_state=seed).split(X, y):
        model = lgb.LGBMClassifier(**LGB_PARAMS, random_state=seed)
        model.fit(X.iloc[tr_idx], y[tr_idx], eval_set=[(X.iloc[va_idx], y[va_idx])],
                  eval_metric="auc", callbacks=[lgb.early_stopping(200, verbose=False)])
        oof_seed[va_idx] = model.predict_proba(X.iloc[va_idx])[:, 1]
        pred_lgb += model.predict_proba(X_test)[:, 1] / (5 * len(LGB_SEEDS))  # mean over 15 models
    oof_lgb += oof_seed / len(LGB_SEEDS)
    print(f"LGB seed {seed}: OOF AUC = {roc_auc_score(y, oof_seed):.5f}  ({time.time() - t:.0f}s)")
print(f"LGB averaged OOF AUC = {roc_auc_score(y, oof_lgb):.5f}")

LGB seed 0: OOF AUC = 0.76279  (98s)


LGB seed 1: OOF AUC = 0.76270  (98s)


LGB seed 2: OOF AUC = 0.76329  (86s)
LGB averaged OOF AUC = 0.76430


## 3. CatBoost — native categorical handling, 5-fold Stratified CV
CatBoost reads text categories directly, so missing values are filled with the word "missing" first.

In [6]:
# CatBoost needs plain text (no NaN) in the categorical columns
X_cb, X_test_cb = X.copy(), X_test.copy()
for c in CAT_COLS:
    X_cb[c] = X_cb[c].astype(object).fillna("missing").astype(str)
    X_test_cb[c] = X_test_cb[c].astype(object).fillna("missing").astype(str)

oof_cat = np.zeros(len(X))
pred_cat = np.zeros(len(X_test))
for fold, (tr_idx, va_idx) in enumerate(StratifiedKFold(5, shuffle=True, random_state=42).split(X_cb, y)):
    t = time.time()
    model = CatBoostClassifier(iterations=5000, learning_rate=0.04, depth=6, l2_leaf_reg=5,
                               eval_metric="AUC", random_seed=42, od_type="Iter", od_wait=200,
                               cat_features=CAT_COLS, verbose=0)
    model.fit(X_cb.iloc[tr_idx], y[tr_idx], eval_set=(X_cb.iloc[va_idx], y[va_idx]), use_best_model=True)
    oof_cat[va_idx] = model.predict_proba(X_cb.iloc[va_idx])[:, 1]
    pred_cat += model.predict_proba(X_test_cb)[:, 1] / 5  # mean over 5 models
    print(f"Cat fold {fold}: AUC = {roc_auc_score(y[va_idx], oof_cat[va_idx]):.5f}  ({time.time() - t:.0f}s)")
print(f"CatBoost OOF AUC = {roc_auc_score(y, oof_cat):.5f}")

Cat fold 0: AUC = 0.76150  (151s)


Cat fold 1: AUC = 0.76540  (210s)


Cat fold 2: AUC = 0.76639  (140s)


Cat fold 3: AUC = 0.76102  (129s)


Cat fold 4: AUC = 0.76364  (145s)
CatBoost OOF AUC = 0.76355


## 4. XGBoost — a third model
XGBoost grows shallow trees (depth 4), so it makes different mistakes from LightGBM and CatBoost.
Models with different mistakes blend better. It uses the same plan: 5-fold stratified CV,
early stopping on the validation fold, and out-of-fold (OOF) predictions.

In [7]:
import xgboost as xgb

XGB_PARAMS = dict(n_estimators=20000, learning_rate=0.03, max_depth=4, min_child_weight=50,
                  subsample=0.8, colsample_bytree=0.3, reg_lambda=20, tree_method="hist",
                  enable_categorical=True, max_cat_to_onehot=8, eval_metric="auc",
                  early_stopping_rounds=200, random_state=0, n_jobs=-1)

oof_xgb = np.zeros(len(X))
pred_xgb = np.zeros(len(X_test))
for fold, (tr_idx, va_idx) in enumerate(StratifiedKFold(5, shuffle=True, random_state=100).split(X, y)):
    t = time.time()
    model = xgb.XGBClassifier(**XGB_PARAMS)
    model.fit(X.iloc[tr_idx], y[tr_idx], eval_set=[(X.iloc[va_idx], y[va_idx])], verbose=False)
    oof_xgb[va_idx] = model.predict_proba(X.iloc[va_idx])[:, 1]
    pred_xgb += model.predict_proba(X_test)[:, 1] / 5  # mean over 5 models
    print(f"XGB fold {fold}: AUC = {roc_auc_score(y[va_idx], oof_xgb[va_idx]):.5f}  ({time.time() - t:.0f}s)")
print(f"XGBoost OOF AUC = {roc_auc_score(y, oof_xgb):.5f}")

XGB fold 0: AUC = 0.76473  (54s)


XGB fold 1: AUC = 0.76115  (50s)


XGB fold 2: AUC = 0.76221  (55s)


XGB fold 3: AUC = 0.76752  (73s)


XGB fold 4: AUC = 0.75899  (53s)
XGBoost OOF AUC = 0.76293


## 5. Rank-average blend of the three models
AUC depends only on the ordering of predictions, so every model is converted to ranks before averaging.
The weights come from a small grid search on the OOF predictions only (the leaderboard is never used).

In [8]:
def to_rank(p):
    """Turn predictions into ranks between 0 and 1."""
    return rankdata(p) / len(p)

r_lgb, r_cat, r_xgb = to_rank(oof_lgb), to_rank(oof_cat), to_rank(oof_xgb)
print(f"equal weights (1/3 each): OOF AUC = {roc_auc_score(y, (r_lgb + r_cat + r_xgb) / 3):.5f}")

# Grid search: weights for LightGBM and CatBoost, XGBoost gets the rest
best_auc, best_w = 0, None
for w_lgb in np.arange(0, 1.01, 0.1):
    for w_cat in np.arange(0, 1.01 - w_lgb, 0.1):
        w_xgb = max(0.0, 1 - w_lgb - w_cat)
        auc = roc_auc_score(y, w_lgb * r_lgb + w_cat * r_cat + w_xgb * r_xgb)
        if auc > best_auc:
            best_auc, best_w = auc, (w_lgb, w_cat, w_xgb)
W_LGB, W_CAT, W_XGB = best_w
print(f"best weights: LGB={W_LGB:.1f}, Cat={W_CAT:.1f}, XGB={W_XGB:.1f}")

oof_blend = W_LGB * r_lgb + W_CAT * r_cat + W_XGB * r_xgb
pred = W_LGB * to_rank(pred_lgb) + W_CAT * to_rank(pred_cat) + W_XGB * to_rank(pred_xgb)
print(f"\nFINAL blend OOF AUC = {roc_auc_score(y, oof_blend):.5f}")

equal weights (1/3 each): OOF AUC = 0.76557


best weights: LGB=0.4, Cat=0.4, XGB=0.2

FINAL blend OOF AUC = 0.76567


## 6. Save submission
The file must have the same columns as `sample_submission.csv`: `SK_ID_CURR` and `TARGET`.

In [9]:
# Check size and that all values are finite numbers
assert len(pred) == len(sample_sub) and np.isfinite(pred).all()
sample_sub["TARGET"] = pred
output_dir = current_dir / "output"
os.makedirs(output_dir, exist_ok=True)
output_file = output_dir / "submission.csv"
sample_sub.to_csv(output_file, index=False)
print(output_file, sample_sub.shape)
sample_sub.head()

/tmp/claude-0/-home-claude/bf821de9-603d-565b-af9b-db23da60b52d/scratchpad/w/output/submission.csv (61500, 2)


,SK_ID_CURR,TARGET
0,171202,0.293828
1,171203,0.936754
2,171204,0.844715
3,171205,0.744000
4,171206,0.913850


## 7. ZIP for Honors (save the notebook first, then run this cell)

In [10]:
# Save this notebook with exactly this name (File > Save a copy in Drive > rename), then run this cell
notebook_filename = "tutorial_final_v2.ipynb"
zip_output_path = current_dir / "tutorial.zip"
notebook_path = current_dir / notebook_filename
if notebook_path.exists():
    with zipfile.ZipFile(zip_output_path, "w", zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(notebook_path, arcname=notebook_filename)
    print("Successfully created:", zip_output_path)
else:
    print("Notebook not found:", notebook_path)
    print("Notebooks in this folder:", [p.name for p in current_dir.glob("*.ipynb")])

Successfully created: /tmp/claude-0/-home-claude/bf821de9-603d-565b-af9b-db23da60b52d/scratchpad/w/tutorial.zip
